# Solution D1 · Reading a PCA: loadings, explained variance and relationship with the diagnosis

**Type**: Interpret

## Objective

Interpret a PCA on 30 measurements of breast nodules: how much variance each component captures, which variables define it (loadings), why the first component is almost a "general factor", how the explained variance links to the redundancy between variables
(three nearly identical size measurements) and whether the first component separates benign from malignant tumors.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
names, y = list(data.feature_names), data.target
Z = StandardScaler().fit_transform(data.data)
cv = StratifiedKFold(5, shuffle=True, random_state=0)

## Commented solution

In [ ]:
pca = PCA().fit(Z)
ratio = pca.explained_variance_ratio_
cumulative = np.cumsum(ratio)
k_for_threshold = {s: int(np.searchsorted(cumulative, s)) + 1 for s in (0.8, 0.9, 0.95)}
print("variance share of the first 5 components:", ratio[:5].round(3), "| components per threshold:", k_for_threshold)

In [ ]:
l1 = pd.Series(pca.components_[0], index=names)
top_pc1 = list(l1.abs().nlargest(5).index)
pc1_sign_share = float((np.sign(l1) == np.sign(l1.sum())).mean())
print("variables with the largest weight in PC1:", top_pc1)
print(f"PC1 loadings with the same sign as the sum: {pc1_sign_share:.0%}")

In [ ]:
columns = [names.index(c) for c in ["mean radius", "mean perimeter", "mean area"]]
size_corr = np.corrcoef(Z[:, columns].T).min()
three_sizes_ratio = PCA().fit(Z[:, columns]).explained_variance_ratio_[0]
print(f"minimum correlation between radius, perimeter and area: {size_corr:.3f} | a PCA on those three columns: the first component explains {three_sizes_ratio:.1%}")

In [ ]:
scores = pca.transform(Z)
diagnosis_corr = np.abs([np.corrcoef(scores[:, i], y)[0, 1] for i in range(6)])
most_correlated_pc = int(diagnosis_corr.argmax()) + 1
acc_k = {k: cross_val_score(make_pipeline(PCA(k), LogisticRegression(max_iter=2000)), Z, y, cv=cv).mean() for k in [1, 2, 5, 10, 30]}
print("correlation with the diagnosis of the first 6 components:", diagnosis_corr.round(2))
print("accuracy with k components:", {k: round(v, 3) for k, v in acc_k.items()})

## Check

In [ ]:
with verify():
    check(close(ratio, pca.explained_variance_ratio_) and 0.42 < ratio[0] < 0.46 and 0.18 < ratio[1] < 0.2, "the first component explains 44% of the variance, the second 19%", "explained_variance_ratio_")
    check(k_for_threshold == {0.8: 5, 0.9: 7, 0.95: 10}, "5 components for 80%, 7 for 90%, 10 for 95% (out of 30 variables)", "np.searchsorted(np.cumsum(ratio), threshold) + 1")
    check(len(top_pc1) == 5 and "mean concave points" in top_pc1 and pc1_sign_share > 0.85, "the variables with the largest weight describe shape and concavity; almost all variables contribute with the same sign", "np.abs(pca.components_[0]) sorted, and the sign of each loading")
    check(size_corr > 0.98 and three_sizes_ratio > 0.99, "radius, perimeter and area are almost the same variable: a single component explains over 99%", "minimum correlation between the three columns and PCA on those three")
    check(len(diagnosis_corr) == 6 and most_correlated_pc == 1 and diagnosis_corr[0] > 0.75 and (diagnosis_corr[1:] < 0.25).all(), "only the first component is linked to the diagnosis (correlation over 0.75); the others below 0.25", "np.corrcoef(scores[:, i], y)")
    check(set(acc_k) == {1, 2, 5, 10, 30} and acc_k[1] > 0.85 and acc_k[10] > acc_k[1] and acc_k[30] > 0.95, "a single component already gives over 0.85; with 10 components it improves; with 30 over 0.95", "LogisticRegression on PCA(k) in a pipeline")
done()

## What you learned

- **The first component is a "general factor":** it explains 44% of the variance and **all 30** variables have the same sign in the loadings (the strongest are the concavity and shape measurements). "Larger and more irregular" tumors have higher values in almost all the measurements:
  it is exactly the axis that separates benign from malignant (correlation with the diagnosis 0.79, against less than 0.2 for all the others).
- **Redundancy lowers the effective dimensionality:** radius, perimeter and area (means) are correlated over 0.98 and are in practice a single variable (a PCA on them gives 99% with one component). With 10 measurements repeated in three versions (mean, error, worst),
  in 30 columns there are many near-copies: hence 5 components for 80% and 10 for 95%.
- **One component is enough for 91%:** in a simple problem like this one a single component already gives a good classifier (0.91), with 2 you reach 0.95 and with 10 0.98, equal to the one with all 30 variables (0.979). It is a case where the explained variance and the usefulness for the target coincide (contrast with C1).
- Loadings are read in groups: variables with the same sign and similar weight "move together". The **sign** of a component is arbitrary: saying "PC1 is high in malignant tumors" or "low" depends on the convention of that run; what matters is the structure of the weights, not the absolute sign.

## Common pitfall

Giving a name to a component ("the severity component") as if it were a measured variable. It is a linear combination that summarizes the variance, and with different data (or another scaler) the same component can change composition.

## Interview question

> *Click on a question to show its answer.*

<details><summary>How do you interpret the principal components?</summary>

I look at the explained variance (scree plot) and at the loadings: which variables have a high weight and with which sign, so I see which ones move together. I verify the relationship with external variables (for example the correlation of the scores with the target), check the stability (bootstrap) and
don't read the sign in absolute terms. I remember that a component is a direction of maximum variance and not a causal factor.

</details>